# Tangram IA — entrenar el detector de fichas por forma (versión Drive)

| qué | dónde |
|---|---|
| paquete de código | `MyDrive/tangram_sintetico.zip` |
| fotos reales para medir | `MyDrive/figuras_armadas_unet.zip` |
| corridas de entrenamiento | `MyDrive/Tangram_YOLO_runs/` |
| pesos finales | `MyDrive/tangram_formas_v2.pt` |

**Ejecuta las celdas en orden, de arriba abajo, sin saltarte ninguna.** Varias
definen variables que las siguientes usan; si el entorno se reinicia, vuelve a
empezar por la 1.

---

### Qué salió mal en la corrida anterior, y qué cambió aquí

La corrida del 1 de septiembre no entrenó nada aprovechable. Cuatro fallos
encadenados, los cuatro ya cerrados:

1. **El entorno estaba en CPU.** `nvidia-smi` no existía y PyTorch cargó la
   versión `+cpu`. Ahora la primera celda **detiene el notebook** si no hay GPU,
   en vez de dejarlo seguir.
2. **La celda de entrenar falló por dos flags que `entrenar.py` no aceptaba**
   (`--patience`, `--workers`). Ya los acepta, así que el entrenamiento nunca
   llegó a arrancar y eso ya no puede repetirse.
3. **La celda de reanudar entrenó sobre COCO, no sobre el Tangram.** Encontró un
   `last.pt` viejo en Drive sin estado de optimizador; Ultralytics avisó por
   consola, empezó un entrenamiento nuevo y —sin `data`— cayó a su dataset por
   defecto, `coco8-seg`: 100 épocas sobre 8 fotos de personas y perros, todas las
   métricas en cero, guardadas en disco local. Ahora se reanuda con
   `entrenar.py --reanudar`, que comprueba el checkpoint antes de tocarlo.
4. **Las rutas de la evaluación estaban mal**: `evaluar_deteccion.py` vive dentro
   del código desempaquetado, no en `/content`; y el zip de test es
   `figuras_armadas_unet.zip`, no `FirgurasArmadas_YOLOv8.zip` —ese otro es un
   dataset distinto, de cajas—. Corregidas, y el modelo viejo ahora viaja dentro
   del propio paquete.


## Por qué hay que reentrenar

El modelo anterior se entrenó con un dataset defectuoso, y el defecto estaba en
la geometría de referencia del proyecto: en `tangram_validator.PIEZAS_CANONICAS`
el **romboide estaba definido con el mismo polígono que el cuadrado** —cuatro
lados iguales y cuatro ángulos de 90°—. Como `composicion.py` construye las
fichas sintéticas a partir de ahí, **las 6000 imágenes de entrenamiento tenían
dos cuadrados y ningún romboide**: el modelo nunca vio la ficha que ahora se le
pide reconocer.

Se notaba en la medición contra fotos reales: le faltaba el romboide en 58 de
114 fotos y el cuadrado en 56, y le sobraban justo los triángulos pequeños que
los sustituían. No los confundía —nunca había visto uno—.

Ya está corregido y verificado: las siete fichas teselan el cuadrado con
cobertura exacta y solape cero, el cuadrado tiene cuatro ángulos de 90° y el
romboide 45/135. La comprobación del validador ahora revisa **ángulos**, no solo
áreas, que es lo que dejaba pasar el error: cuadrado y romboide miden lo mismo.

**Sube el `tangram_sintetico.zip` nuevo antes de correr esto** —el de 35,9 MB con
fecha de hoy—, o vas a regenerar el mismo dataset defectuoso. La corrida se llama
`tangram_formas_v2` para no pisar la anterior, que queda como referencia.

## 0. Comprobar que hay GPU

Si esta celda para, ve a **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**
y vuelve a ejecutarla. No sigas sin GPU: el entrenamiento pasa de unas 2-3 horas a
varios días, y no se nota hasta que llevas una hora esperando.


In [ ]:
import subprocess, sys

try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or '(sin salida)')
except FileNotFoundError:
    raise SystemExit(
        'No hay GPU en este entorno.\n'
        'Entorno de ejecucion -> Cambiar tipo de entorno -> GPU (T4), y repite esta celda.'
    )

In [ ]:
!pip install -q ultralytics
import torch, ultralytics
ultralytics.checks()

assert torch.cuda.is_available(), (
    'PyTorch no ve la GPU aunque nvidia-smi responda. Reinicia el entorno '
    '(Entorno de ejecucion -> Reiniciar entorno) y repite desde la celda 0.'
)
print('GPU vista por PyTorch:', torch.cuda.get_device_name(0))

## 1. Montar Drive y desempaquetar el código

El zip se descomprime en `/content` (disco local): el código se lee miles de veces
durante el entrenamiento y desde Drive iría lento.

Dos cosas que esta celda hace y que ahorran dolores de cabeza:

- **Comprueba que Drive responda de verdad.** Colab contesta *"already mounted"*
  aunque el montaje se haya caído, y a partir de ahí cualquier lectura falla con
  `Transport endpoint is not connected`. La única forma de saberlo es intentar
  listar la carpeta, así que eso hace: si no responde, lo remonta solo.
- **Descomprime con Python, no con `unzip`.** `unzip -q` falla en silencio y deja
  el error para dos celdas después.

Además hace `chdir` al código desempaquetado y **deja ahí el directorio de
trabajo para todo el notebook**: es lo que permite que `python -m sintetico...` y
`python evaluar_deteccion.py` encuentren sus módulos. La corrida anterior falló
justo por esto.


In [ ]:
import os, zipfile
from pathlib import Path
from google.colab import drive

DRIVE   = '/content/drive/MyDrive'
ZIP     = f'{DRIVE}/tangram_sintetico.zip'
ZIP_TEST= f'{DRIVE}/figuras_armadas_unet.zip'   # las 114 fotos reales con su mascara
RUNS    = f'{DRIVE}/Tangram_YOLO_runs'          # las corridas viven en Drive: sobreviven a una desconexion
CODIGO  = '/content/tangram_sintetico'          # disco local
DATASET = '/content/ds'                         # disco local, NO Drive
NOMBRE  = 'tangram_formas_v2'   # v2: primer entrenamiento con el romboide corregido


def montar_drive():
    """Monta Drive comprobando que de verdad responde.

    Colab responde 'already mounted' aunque el montaje se haya caido, y entonces
    cualquier lectura revienta con 'Transport endpoint is not connected'. La unica
    forma de saberlo es intentar listar la carpeta.
    """
    try:
        os.listdir(DRIVE)
        print('Drive operativo.')
        return
    except Exception:
        pass
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)
    os.listdir(DRIVE)
    print('Drive montado.')


montar_drive()

faltan = [z for z in (ZIP, ZIP_TEST) if not os.path.exists(z)]
if faltan:
    print('En la raiz de tu MyDrive hay estos zip:')
    for z in sorted(Path(DRIVE).glob('*.zip')):
        print(f'   {z.name:44s} {z.stat().st_size/1048576:7.1f} MB')
    raise SystemExit('Faltan en MyDrive: ' + ', '.join(os.path.basename(z) for z in faltan))

with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content')

os.chdir(CODIGO)          # <- todo el notebook trabaja desde aqui
os.makedirs(RUNS, exist_ok=True)

NECESARIOS = ('sintetico/generar.py', 'sintetico/entrenar.py',
              'evaluar_deteccion.py', 'tangram_validator.py')
faltan = [n for n in NECESARIOS if not os.path.exists(n)]
if faltan:
    import datetime
    with zipfile.ZipFile(ZIP) as z:
        dentro = z.namelist()
    tam   = os.path.getsize(ZIP) / 1048576
    fecha = datetime.datetime.fromtimestamp(os.path.getmtime(ZIP)).strftime('%d-%m-%Y %H:%M')
    print(f'{ZIP}\n  {tam:.1f} MB, subido el {fecha}\n  contenido (sin fondos_reales):')
    for x in sorted(x for x in dentro if 'fondos_reales' not in x):
        print('   ', x)
    raise SystemExit(
        'Al zip de tu Drive le faltan: ' + ', '.join(faltan) + '\n\n'
        'Es la version antigua del paquete. La nueva pesa ~36 MB (la vieja, 15) y trae\n'
        'el codigo dentro de sintetico/, mas evaluar_deteccion.py, tangram_validator.py\n'
        'y models/tangram_piezas_seg_best.pt.\n\n'
        'Sube a la raiz de tu MyDrive, reemplazando el que hay:\n'
        '  ...\\tangram-ia\\vision-service\\tangram_sintetico.zip\n\n'
        'Drive no reemplaza en el sitio: si te deja "tangram_sintetico (1).zip",\n'
        'borra el viejo primero y renombra el nuevo. Luego repite esta celda.'
    )

print('\nCodigo en', os.getcwd())
print(sorted(os.listdir('.')))

## 1b. Revisar qué hay ya en `Tangram_YOLO_runs`

Esta celda es nueva y existe por el fallo 3. Antes de entrenar, mira qué hay en la
carpeta de la corrida y **dice sobre qué datos se entrenó**. Si aparece
`coco8-seg` o clases como `person` / `dog`, esos pesos son los de la corrida
fallida: no sirven y hay que apartarlos antes de seguir.

No borra nada por su cuenta.


In [ ]:
import torch
from pathlib import Path

corrida = Path(RUNS) / NOMBRE
ultimo  = corrida / 'weights' / 'last.pt'

if not ultimo.exists():
    print(f'{corrida} esta limpia. Adelante con la seccion 2.')
else:
    ck = torch.load(ultimo, map_location='cpu', weights_only=False)
    datos_prev = (ck.get('train_args') or {}).get('data')
    clases     = list(getattr(ck.get('model'), 'names', {}).values())
    epoca      = ck.get('epoch', -1)

    print(f'Hay una corrida en {corrida}')
    print(f'  entrenada sobre : {datos_prev}')
    print(f'  clases          : {clases[:8]}{" ..." if len(clases) > 8 else ""}')
    print(f'  epoca guardada  : {epoca}   (-1 = entrenamiento ya cerrado, NO reanudable)')

    sospechosa = (datos_prev and 'coco' in str(datos_prev).lower()) or \
                 any(c in clases for c in ('person', 'dog', 'horse'))
    if sospechosa:
        print('\n  >>> Estos pesos son de la corrida fallida (COCO, no Tangram).')
        print('  >>> Apartalos con la linea de abajo y vuelve a ejecutar esta celda.')
        print(f"\n      !mv '{corrida}' '{corrida}_FALLIDA_coco8'")
    elif epoca < 0:
        print('\n  Entrenamiento terminado. Para uno nuevo, cambia NOMBRE arriba.')
    else:
        print(f'\n  Reanudable desde la epoca {epoca}: usa la seccion 6, no la 5.')

## 2. Generar el dataset

6000 imágenes de entrenamiento y 800 de validación, a 640 px. En la CPU de Colab
son unos **25-35 minutos** (esto se genera en CPU aunque el entrenamiento vaya en
GPU). Si quieres probar el circuito completo primero, baja a `--train 600 --val 100`:
sale en 3 minutos y el resto del notebook es idéntico.

Las 7 fichas salen juntas y compartiendo aristas, con color y material sorteados en
cada muestra. Las anotaciones son exactas por construcción: se anota el mismo
polígono que se dibuja.

La semilla queda fija en 1234. **Anótalo:** si la sesión se cae y hay que regenerar,
con la misma semilla sale el mismo dataset y el entrenamiento puede reanudarse.


In [ ]:
!python -m sintetico.generar --salida "{DATASET}" --train 6000 --val 800 --lado 640 --semilla 1234

## 3. Mirar las muestras

**No te saltes esto.** Un dataset puede estar numéricamente perfecto y ser
visualmente inservible, y eso no lo detecta ninguna métrica.


In [ ]:
from IPython.display import Image, display
display(Image(f'{DATASET}/muestras.jpg'))

## 4. Verificar las anotaciones

Comprueba el formato **y** pasa las anotaciones por el validador geométrico del
proyecto. Si él ve en cada imagen un Tangram completo, sin fichas montadas ni
sueltas, los datos son coherentes con el sistema que va a consumirlos.

Referencia de lo que salió la vez anterior, y que debería repetirse:
solape entre fichas ≈ 0.010 (tolerado 0.05), hueco interior ≈ 0.002.


In [ ]:
!python -m sintetico.verificar "{DATASET}" --muestreo 20

## 5. Entrenar

Los pesos caen en `MyDrive/Tangram_YOLO_runs/tangram_formas/`. Ultralytics escribe
`last.pt` en cada época, así que a partir de aquí una desconexión ya no cuesta el
trabajo entero.

`--workers 2` porque Colab da 2 vCPU: pedir 8 procesos de carga en 2 núcleos los
deja peleándose entre ellos.

`hsv_h=0.5`, dentro de `entrenar.py`, es lo que impide que el modelo vuelva a
apoyarse en el color: rota el matiz por todo el círculo cromático en cada época.
No lo bajes — es el propósito entero de este entrenamiento.

Si ya existe una corrida con este nombre, `entrenar.py` **para en vez de
sobrescribirla**. Para continuarla, salta a la sección 6.


In [ ]:
!python -m sintetico.entrenar \
    --dataset "{DATASET}" \
    --epocas 100 --imgsz 640 --batch 16 --device 0 \
    --salida "{RUNS}" --nombre "{NOMBRE}" \
    --patience 25 --workers 2

## 6. Si Colab se desconectó

Vuelve a ejecutar las celdas de las secciones 0 a 2 (GPU, Drive, **regenerar el
dataset con la misma semilla**) y luego esta.

`--reanudar` no es el `resume=True` pelado de antes: antes de tocar nada comprueba
que el checkpoint lleve estado de optimizador y que se entrenara con este mismo
`data.yaml`. Si algo no cuadra, para y lo dice. Es la guarda que faltaba cuando el
notebook se puso a entrenar sobre COCO.


In [ ]:
!python -m sintetico.entrenar \
    --dataset "{DATASET}" \
    --epocas 100 --imgsz 640 --batch 16 --device 0 \
    --salida "{RUNS}" --nombre "{NOMBRE}" \
    --patience 25 --workers 2 \
    --reanudar

## 7. Resultados del entrenamiento

`results.png` muestra las curvas de pérdida y mAP. Lo que interesa es que
`mAP50-95(M)` —la de máscaras— suba y se estabilice, y que las pérdidas de
validación no se despeguen de las de entrenamiento.

La celda imprime además el mejor mAP alcanzado. **Si sale 0, el entrenamiento no
aprendió nada** y no tiene sentido pasar a la sección 8: revisa antes las muestras
de la sección 3.


In [ ]:
from IPython.display import Image, display
import pandas as pd
from pathlib import Path

base = Path(RUNS) / NOMBRE
for fig in ('results.png', 'confusion_matrix_normalized.png'):
    if (base / fig).exists():
        display(Image(str(base / fig)))

csv = base / 'results.csv'
if csv.exists():
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]
    col = next((c for c in df.columns if 'mAP50-95(M)' in c), None)
    if col:
        print(f'Mejor mAP50-95 de mascara: {df[col].max():.4f}  (epocas completadas: {len(df)})')
        if df[col].max() == 0:
            print('AVISO: cero. El modelo no aprendio nada; no sigas a la seccion 8.')

## 8. La medición que de verdad importa

El `val` de arriba es sintético igual que el entrenamiento: dice que el modelo
converge, **no** que funcione sobre una mesa. Esto sí lo dice.

`figuras_armadas_unet.zip` son las 114 fotos reales con la máscara de la silueta.
Se compara la unión de las máscaras detectadas contra la silueta real.

**La línea base a batir**, medida con el modelo que hoy está en producción
(`tangram_piezas_seg_best.pt`, que viaja dentro del propio paquete de código, así
que no hay que buscarlo en Drive):

| | `tangram_piezas_seg_best.pt` |
|---|---|
| IoU de silueta (media) | 0.096 |
| IoU ≥ 0.75 | 2 de 114 (2 %) |
| Fichas detectadas | 1.21 de 7 |
| Fotos con las 7 | 2 de 114 (2 %) |


In [ ]:
import os, zipfile
from pathlib import Path

os.chdir(CODIGO)                      # por si el entorno se reinicio

with zipfile.ZipFile(ZIP_TEST) as z:
    z.extractall('/content/unet_ds')

RUTA_TEST = Path('/content/unet_ds')
if not (RUTA_TEST / 'train').is_dir():          # el zip trae carpeta interna
    hijos = [d for d in RUTA_TEST.iterdir() if d.is_dir() and (d / 'train').is_dir()]
    assert hijos, f'No encuentro train/val/test dentro de {RUTA_TEST}'
    RUTA_TEST = hijos[0]

print('dataset de test:', RUTA_TEST)
for split in ('train', 'val', 'test'):
    d = RUTA_TEST / split / 'images'
    print(f'  {split}: {len(list(d.glob("*"))) if d.is_dir() else 0} imagenes')

In [ ]:
import shlex, sys
from pathlib import Path

nuevos = Path(RUNS) / NOMBRE / 'weights' / 'best.pt'
viejos = Path(CODIGO) / 'models' / 'tangram_piezas_seg_best.pt'
if not viejos.exists():                       # por si se usa un zip antiguo
    viejos = Path(DRIVE) / 'tangram_piezas_seg_best.pt'
assert nuevos.exists(), f'No hay pesos entrenados en {nuevos}. Corre la seccion 5.'

cmd = [sys.executable, 'evaluar_deteccion.py', str(RUTA_TEST),
       '--splits', 'train,val,test',
       '--csv', '/content/comparacion.csv']
if viejos.exists():
    # el viejo primero y el nuevo en --comparar: asi la tabla sale en ese orden
    cmd += ['--pesos', str(viejos), '--comparar', str(nuevos)]
else:
    print('AVISO: falta el modelo viejo; se mide solo el nuevo, sin comparativa.\n')
    cmd += ['--pesos', str(nuevos)]

orden = ' '.join(shlex.quote(c) for c in cmd)
print(orden, '\n')
# subprocess.run() manda la salida al log del kernel y la celda queda vacia;
# hay que pasar la orden por la shell de IPython para verla aqui.
get_ipython().system(orden)

## 9. Guardar los pesos y desplegar

Se copian a la raíz de MyDrive con la misma convención de siempre. La celda
comprueba que cada archivo exista antes de copiarlo: la versión anterior decía
"Listo" aunque las dos copias hubieran fallado.


In [ ]:
import shutil
from pathlib import Path

destinos = [
    (Path(RUNS) / NOMBRE / 'weights' / 'best.pt', Path(DRIVE) / f'{NOMBRE}.pt'),
    (Path('/content/comparacion.csv'),            Path(DRIVE) / f'{NOMBRE}_comparacion.csv'),
]

for origen, destino in destinos:
    if origen.exists():
        shutil.copy(origen, destino)
        print(f'copiado  {destino.name}  ({destino.stat().st_size/1048576:.1f} MB)')
    else:
        print(f'FALTA    {origen}  -> no se copio nada')

### En tu PC

1. Baja `tangram_formas_v2.pt` de Drive a `vision-service/models/`
2. En `vision-service/.env`: `YOLO_WEIGHTS=models/tangram_formas_v2.pt`
3. Reinicia el servicio y comprueba en `/health` que `yolo_loaded` sea `true`

No hay que tocar el backend ni la app: las clases son las cinco geométricas que
`tangram_validator.resolver_taxonomia()` ya reconoce.

### Y después

Con el modelo nuevo puesto, toca recalibrar el umbral del validador:

```
python tangram_validator.py --calibrar
```

y ajustar `MATCH_IOU` / `CLOSE_IOU` en el `.env` con ese resultado, no con el 0.75
que está hoy.
